# 05 - Two-Tier Memory with Progressive Close-ups (training-free, B only)
```text
Offline, per video:   SigLIP2 embeddings of all frames --> K-means (K grows with video length) --> K clusters
                      (representative = frame nearest the centre, sharp frames preferred)

Per question:
  B reads the question --> task type
  Round 1:  SUMMARY    K low-res frames (320x180), one per cluster, in time order + "scene visible ~X s"
            CLOSE-UPS  8 (clip < 1 min) or 16 high-res frames (640x360)
                       - object / spatial / event: top clusters ranked by size x question relevance
                       - counting / temporal: spread over time (gap filling); clusters would merge repeated events
            --> B answers + confidence.  confident (>= tau) --> STOP
  Round 2-3: B writes a MEMORY NOTE (evidence + what is still unclear, not its answer)
            --> same summary + note + 8 NEW close-ups (unseen clusters / largest time gaps,
                steered by the note's "LOOK FOR" line)
            --> B answers.  confident, or same answer as the previous round --> STOP
```

### Sections
1. Configuration
2. Data
3. Clustering
4. Close-up selection
5. Prompts and model (B)
6. The pipeline with memory and stopping
7. Runner
8. Calibrate τ (calibration split)
9. Full method on the test split
10. Baselines B1-B6
11. Results
12. Embedding visualization (PCA / t-SNE)

In [ ]:
import os
import re
import io
import gc
import json
import math
import time
import hashlib
import functools
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Configuration

In [ ]:
# ------------------------------------------------------------------ paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
SUBSET_DIR = DATA_ROOT / "subset"
FRAMES_ROOT = DATA_ROOT / "frames"
FEATURES_ROOT = DATA_ROOT / "features"
RESULTS_DIR = PROJECT_ROOT / "results" / "two_tier"
NB04_DIR = PROJECT_ROOT / "results" / "proposed"         # notebook 04 results, shown for reference if present
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------ data
FPS = {"mvbench": 5.0, "videomme": 1.0}                  # must match notebooks 01 / 03
DATASETS = ["mvbench", "videomme"]
LIMIT_PER_CATEGORY = None                                 # small int for a smoke test
SEED = 42

# ------------------------------------------------------------------ clustering: K = clip(round(K_SCALE * log2(1 + seconds/10)), K_MIN, K_MAX)
K_SCALE, K_MIN, K_MAX = 8.0, 8, 48                        # 10 s -> 8, 1 min -> 22, 5 min -> 40, 15 min -> 48

# ------------------------------------------------------------------ resolutions (fit inside the box, keep aspect ratio, never upscale)
SUMMARY_BOX = (320, 180)                                  # ~66 visual tokens per frame on Qwen3-VL
CLOSEUP_BOX = (640, 360)                                  # ~220 visual tokens per frame

# ------------------------------------------------------------------ close-ups and rounds
CLOSEUPS_SHORT, CLOSEUPS_LONG, SHORT_CLIP_S = 8, 16, 60   # round-1 close-ups: 8 for clips < 60 s, else 16
CLOSEUPS_PER_ROUND = 8                                    # new close-ups added in rounds 2 and 3
CLOSEUP_CAP = 32                                          # max close-ups over all rounds
MAX_ROUNDS = 3
RANK = "size_x_rel"                                       # cluster ranking: "size_x_rel" | "size" (Flash-VStream style) | "rel"
STABILITY_STOP = True                                     # also stop when the answer is unchanged from the previous round
MEMORY_MAX_WORDS = 120
TAU_DEFAULT = 0.90                                        # used only until section 8 has produced a calibrated tau
TAU_TOLERANCE = 0.01                                      # calibration: cheapest tau within 1 pt of the best accuracy

# ------------------------------------------------------------------ model (B only)
QWEN_B = "Qwen/Qwen3-VL-4B-Instruct"
QWEN_B_4BIT = False
SIGLIP_ID = "google/siglip2-base-patch16-224"             # text encoder for the memory note's LOOK FOR query

## 2. Data

In [ ]:
LETTERS = "ABCDEFGH"
CATEGORIES = ["temporal_reasoning", "object_recognition", "counting", "spatial_reasoning", "event_understanding"]


def fps_dirname(fps):
    return "fps_" + f"{fps:g}".replace(".", "p")


with np.load(FEATURES_ROOT / "questions.npz") as z:
    _Q = {q: a.astype(np.float32) for q, a in zip(z["qid"], z["q_emb"])}


def load_samples(split="test", datasets=DATASETS, limit=LIMIT_PER_CATEGORY):
    name = {"test": "manifest.jsonl", "calib": "manifest_calib.jsonl"}[split]
    samples, missing = [], 0
    for line in (SUBSET_DIR / name).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        if r["dataset"] not in datasets:
            continue
        fdir = FRAMES_ROOT / fps_dirname(FPS[r["dataset"]]) / r["dataset"] / r["frame_key"]
        fpath = FEATURES_ROOT / fps_dirname(FPS[r["dataset"]]) / r["dataset"] / f"{r['frame_key']}.npz"
        if not (fdir / "frames.json").exists() or not fpath.exists() or r["qid"] not in _Q:
            missing += 1
            continue
        meta = json.loads((fdir / "frames.json").read_text())
        with np.load(fpath) as z:
            feat = {k: z[k] for k in ("t", "emb", "motion", "sharp", "contrast")}
        feat["emb"] = feat["emb"].astype(np.float32)
        r.update(split=split, fps=FPS[r["dataset"]], frame_dir=str(fdir), feat=feat,
                 frames=[dict(f, path=str(fdir / f["file"])) for f in meta["frames"]],
                 span_s=float(feat["t"][-1] - feat["t"][0]) + 1.0 / FPS[r["dataset"]], q_emb=_Q[r["qid"]])
        samples.append(r)
    if limit:
        by = {}
        for s in samples:
            by.setdefault((s["dataset"], s["category"]), []).append(s)
        samples = [s for g in by.values() for s in g[:limit]]
    if missing:
        print(f"WARNING {split}: {missing} questions skipped (frames or features missing - run notebooks 01 and 03)")
    return samples


TEST = load_samples("test")
CALIB = load_samples("calib")
BY_QID = {s["qid"]: s for s in TEST + CALIB}
print(f"test {len(TEST)} | calib {len(CALIB)}")

In [ ]:
def fit_box(im, box):
    s = min(1.0, box[0] / im.width, box[1] / im.height)
    return im if s == 1.0 else im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))), Image.BICUBIC)


@functools.lru_cache(maxsize=2048)
def load_frame(path, box):
    return fit_box(Image.open(path).convert("RGB"), box)


@functools.lru_cache(maxsize=8192)
def frame_bytes(path, box):
    buf = io.BytesIO()
    load_frame(path, box).save(buf, format="JPEG", quality=90)
    return len(buf.getvalue())


def _norm(x):
    x = np.asarray(x, dtype=np.float32)
    lo, hi = np.percentile(x, 5), np.percentile(x, 95)
    return np.clip((x - lo) / (hi - lo + 1e-8), 0, 1)


def signals(s):
    # per-frame signals, normalized per video: relevance to the question, quality (sharpness, blank frames = 0), motion
    if "_sig" not in s:
        f = s["feat"]
        s["_sig"] = dict(rel=_norm(f["emb"] @ s["q_emb"]), q=_norm(np.log1p(f["sharp"])) * (f["contrast"] > 8.0),
                         motion=_norm(f["motion"]), emb=f["emb"], n=len(f["t"]))
    return s["_sig"]

## 3. Clustering

Spherical K-means (cosine similarity) on the SigLIP2 embeddings, with k-means++ initialisation and a fixed seed per
video, so results are reproducible. Empty clusters are dropped. Each cluster keeps:
- its member frames and screen time (members / fps)
- a representative: among the 3 frames nearest the centre, the sharpest

In [ ]:
def n_clusters(s):
    k = round(K_SCALE * math.log2(1 + s["span_s"] / 10.0))
    return int(max(1, min(np.clip(k, K_MIN, K_MAX), len(s["frames"]) // 2)))


def kmeans(x, k, seed, iters=60):
    rng = np.random.default_rng(seed)
    n = len(x)
    cent = [x[rng.integers(n)]]
    dist = 1.0 - x @ cent[0]
    for _ in range(1, k):                                   # k-means++
        p = np.maximum(dist, 0) ** 2
        p = p / p.sum() if p.sum() > 0 else np.full(n, 1.0 / n)
        cent.append(x[rng.choice(n, p=p)])
        dist = np.minimum(dist, 1.0 - x @ cent[-1])
    cent = np.stack(cent)
    for _ in range(iters):
        lab = np.argmax(x @ cent.T, axis=1)
        new = np.stack([x[lab == j].mean(0) if (lab == j).any() else cent[j] for j in range(k)])
        new /= np.linalg.norm(new, axis=1, keepdims=True) + 1e-8
        if np.allclose(new, cent, atol=1e-6):
            break
        cent = new
    return np.argmax(x @ cent.T, axis=1), cent


def clusters(s):
    if "_cl" not in s:
        sig = signals(s)
        k = n_clusters(s)
        seed = int(hashlib.md5(s["frame_dir"].encode()).hexdigest()[:8], 16) ^ SEED
        lab, cent = kmeans(sig["emb"], k, seed)
        members, reps, keep = [], [], []
        for j in range(k):
            m = np.flatnonzero(lab == j)
            if not len(m):
                continue
            near = m[np.argsort(-(sig["emb"][m] @ cent[j]))[:3]]
            reps.append(int(near[np.argmax(sig["q"][near])]))
            members.append(m)
            keep.append(j)
        label = np.full(sig["n"], -1)
        for new_j, m in enumerate(members):
            label[m] = new_j
        order = np.argsort([s["frames"][r]["t"] for r in reps])     # clusters numbered in time order of their representative
        members = [members[i] for i in order]
        reps = [reps[i] for i in order]
        label = np.argsort(order)[label]
        s["_cl"] = dict(members=members, reps=reps, label=label, size=np.array([len(m) for m in members]),
                        seconds=np.array([len(m) / s["fps"] for m in members]))
    return s["_cl"]


st = pd.DataFrame([dict(dataset=s["dataset"], seconds=s["span_s"], frames=len(s["frames"]), K=len(clusters(s)["reps"]))
                   for s in TEST])
st.groupby("dataset")[["seconds", "frames", "K"]].describe().round(1).loc[:, (slice(None), ["mean", "min", "max"])]

## 4. Close-up selection

`pick_closeups(s, task, n, sent, look_for)` returns `n` new frames that haven't been sent yet.

- **counting / temporal_reasoning:** gap filling. Each pick goes into the middle of the **largest time gap** between
  frames already chosen, and the best frame there wins: 0.5 motion + 0.3 relevance + 0.2 quality for counting,
  0.6 relevance + 0.4 quality for temporal. Round 1 therefore spreads over the whole video, and later rounds fill
  the holes. No clusters are used, so repeated events stay separate.
- **object / spatial / event:** clusters are ranked by `RANK`, where `size_x_rel` = (cluster size / largest size) ×
  (best relevance in the cluster). Picks go round-robin: clusters without a close-up come first, taken in rank
  order, and each contributes its best unsent frame (relevance + 0.3 quality).
- **look_for:** in later rounds, relevance is blended 50/50 with SigLIP similarity to B's "LOOK FOR" text.
- Summary representatives are avoided as close-ups when enough other frames exist.

In [ ]:
TIME_TASKS = {"counting", "temporal_reasoning"}


def pick_closeups(s, task, n, sent=(), look_for_emb=None, rank=RANK):
    sig, cl = signals(s), clusters(s)
    rel = sig["rel"] if look_for_emb is None else _norm(0.5 * sig["rel"] + 0.5 * _norm(sig["emb"] @ look_for_emb))
    avail = np.ones(sig["n"], dtype=bool)
    avail[list(sent)] = False
    soft = avail.copy()
    soft[cl["reps"]] = False
    if soft.sum() >= n:                                     # avoid summary frames if there is room
        avail = soft
    n = int(min(n, avail.sum()))
    picked = []
    if task in TIME_TASKS:
        w = (0.5, 0.3, 0.2) if task == "counting" else (0.0, 0.6, 0.4)
        score = w[0] * sig["motion"] + w[1] * rel + w[2] * sig["q"]
        for _ in range(n):
            bounds = [-1] + sorted(set(sent) | set(picked)) + [sig["n"]]
            for a, b in sorted(zip(bounds[:-1], bounds[1:]), key=lambda ab: -(ab[1] - ab[0])):
                inside = [i for i in range(a + 1, b) if avail[i] and i not in picked]
                if not inside:
                    continue
                lo, hi = a + (b - a) / 4, b - (b - a) / 4
                mid = [i for i in inside if lo <= i <= hi] or inside
                picked.append(max(mid, key=lambda i: score[i]))
                break
    else:
        size = cl["size"] / cl["size"].max()
        crel = np.array([rel[m].max() for m in cl["members"]])
        cscore = {"size_x_rel": size * crel, "size": size, "rel": crel}[rank]
        count = np.zeros(len(cl["members"]), dtype=int)
        for i in sent:
            count[cl["label"][i]] += 1
        fscore = rel + 0.3 * sig["q"]
        while len(picked) < n:
            for j in sorted(range(len(cl["members"])), key=lambda j: (count[j], -cscore[j])):
                cand = [i for i in cl["members"][j] if avail[i] and i not in picked]
                if cand:
                    picked.append(int(max(cand, key=lambda i: fscore[i])))
                    count[j] += 1
                    break
            else:
                break
    return sorted(int(i) for i in picked)


def uniform_idx(n_total, k):
    k = min(k, n_total)
    return sorted({int(i) for i in ((np.arange(k) + 0.5) * n_total / k).astype(int)})

## 5. Prompts and model (B)

The final answer uses each benchmark's official prompt wording (same as notebook 04). Every image part carries its
box, so a single call can mix low-res summary frames with high-res close-ups.

The notebook 04 memory fix for Windows (memory-efficient attention for grouped-query attention) is applied here too.

In [ ]:
MVBENCH_SYSTEM = ("Carefully watch the video and pay attention to the cause and sequence of events, the detail and "
                  "movement of objects, and the action and pose of persons. Based on your observations, select the "
                  "best option that accurately addresses the question.")
ANSWER_PREFIX = "Best option: ("

ROUTER_PROMPT = """Classify the question about a video into exactly one category.
(A) temporal reasoning - order or sequence of events, what happened before/after/next
(B) object recognition - what object, person or attribute is present, whether something exists, what someone interacts with
(C) counting - how many things or how many times something happens
(D) spatial reasoning - positions, directions, locations, where things are or move relative to each other
(E) event understanding - why or how something happened, purpose or cause of actions, unexpected events, changes, what would happen if
Question: {question}
Answer with the letter only."""
ROUTER_LABELS = ["temporal_reasoning", "object_recognition", "counting", "spatial_reasoning", "event_understanding"]

MEMORY_PROMPT = f"""You have not settled on an answer yet and will get more close-up frames of this video.
Write a short note for yourself (at most {MEMORY_MAX_WORDS} words):
- what you observed that matters for the question, with frame times;
- what is still unclear or missing.
Do NOT say which option you choose. End with one line in this exact format:
LOOK FOR: <a short description of what the next frames should show>"""


def mc_system(s):
    return MVBENCH_SYSTEM if s["dataset"] == "mvbench" else None


def mc_text(s):
    if s["dataset"] == "videomme":
        opts = "\n".join(f"{LETTERS[i]}. {o}" for i, o in enumerate(s["options"]))
        return ("Select the best answer to the following multiple-choice question based on the video. "
                f"Respond with only the letter ({', '.join(LETTERS[:len(s['options'])])}) of the correct option.\n"
                f"{s['question']}\n{opts}\nThe best answer is:")
    opts = "\n".join(f"({LETTERS[i]}) {o}" for i, o in enumerate(s["options"]))
    return f"Question: {s['question']}\nOptions:\n{opts}\nOnly give the best option."


def build_parts(s, summary=(), closeups=(), memory=None):
    parts = []
    if summary:
        cl = clusters(s)
        parts.append({"type": "text", "text": f"Overview of the whole video: {len(summary)} low-resolution frames, "
                                              "one per distinct scene, in chronological order."})
        rep_sec = {r: cl["seconds"][j] for j, r in enumerate(cl["reps"])}
        for i in summary:
            parts += [{"type": "text", "text": f"Scene at {s['frames'][i]['t']:.1f}s (visible ~{rep_sec.get(i, 0):.0f}s in total):"},
                      {"type": "image", "path": s["frames"][i]["path"], "box": SUMMARY_BOX}]
    if closeups:
        head = "Close-up frames at higher resolution" if summary else "Frames sampled from the video"
        parts.append({"type": "text", "text": f"{head}, in chronological order:"})
        for i in closeups:
            parts += [{"type": "text", "text": f"Frame at {s['frames'][i]['t']:.1f}s:"},
                      {"type": "image", "path": s["frames"][i]["path"], "box": CLOSEUP_BOX}]
    if memory:
        parts.append({"type": "text", "text": f"Your notes from your previous look at this video:\n{memory}"})
    return parts


def parts_bytes(parts):
    return sum(frame_bytes(p["path"], p["box"]) for p in parts if p["type"] == "image")

In [ ]:
def new_usage():
    return {"calls": 0, "in_tokens": 0, "out_tokens": 0, "images": 0}


import transformers.integrations.sdpa_attention as _sdpa
_sdpa.use_gqa_in_sdpa = lambda *args, **kwargs: False       # memory fix, see notebook 04


class QwenAgent:
    def __init__(self, model_id, load_in_4bit=False):
        from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
        kw = dict(dtype=torch.bfloat16, device_map="cuda:0", attn_implementation="sdpa")
        if load_in_4bit:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.bfloat16)
        self.model = AutoModelForImageTextToText.from_pretrained(model_id, **kw).eval()
        self.processor = AutoProcessor.from_pretrained(model_id)
        self.name = model_id.split("/")[-1] + ("-4bit" if load_in_4bit else "")
        tok = self.processor.tokenizer
        self._letter_ids = []
        for L in LETTERS:
            ids = {tok.convert_tokens_to_ids(L)} | {tok.encode(" " + L, add_special_tokens=False)[0]}
            self._letter_ids.append(sorted(i for i in ids if i is not None and i != tok.unk_token_id))
        self.usage = new_usage()

    def _inputs(self, system, parts, prefix=""):
        content, images = [], []
        for p in parts:
            if p["type"] == "text":
                content.append({"type": "text", "text": p["text"]})
            else:
                content.append({"type": "image"})
                images.append(load_frame(p["path"], p.get("box", CLOSEUP_BOX)))
        messages = ([{"role": "system", "content": [{"type": "text", "text": system}]}] if system else []) + \
                   [{"role": "user", "content": content}]
        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False) + prefix
        inputs = self.processor(text=[prompt], images=images or None, return_tensors="pt")
        inputs.pop("attention_mask", None)
        self.usage["calls"] += 1
        self.usage["images"] += len(images)
        self.usage["in_tokens"] += int(inputs["input_ids"].shape[1])
        return inputs.to(self.model.device, dtype=torch.bfloat16)

    @torch.inference_mode()
    def generate(self, system, parts, max_new_tokens=256):
        inputs = self._inputs(system, parts)
        n = inputs["input_ids"].shape[1]
        out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  temperature=None, top_p=None, top_k=None)
        self.usage["out_tokens"] += int(out.shape[1] - n)
        return self.processor.batch_decode(out[:, n:], skip_special_tokens=True)[0].strip()

    @torch.inference_mode()
    def choose(self, system, parts, n_options, prefix=ANSWER_PREFIX):
        inputs = self._inputs(system, parts, prefix)
        logits = self.model(**inputs, logits_to_keep=1).logits[0, -1].float()
        logp = torch.log_softmax(logits, dim=-1)
        per = torch.stack([torch.logsumexp(logp[ids], 0) for ids in self._letter_ids[:n_options]])
        probs = torch.softmax(per, 0).cpu().numpy()
        i = int(np.argmax(probs))
        return dict(pred=LETTERS[i], conf=float(probs[i]), probs=[round(float(p), 4) for p in probs],
                    letter_mass=round(float(per.exp().sum()), 4))

    def route(self, question):
        r = self.choose(None, [{"type": "text", "text": ROUTER_PROMPT.format(question=question)}], 5, prefix="(")
        return ROUTER_LABELS[LETTERS.index(r["pred"])], r["conf"]

    def close(self):
        del self.model
        gc.collect()
        torch.cuda.empty_cache()


# SigLIP2 text encoder: embeds B's "LOOK FOR" line so later rounds can search for it
from transformers import AutoModel, AutoProcessor
_siglip = AutoModel.from_pretrained(SIGLIP_ID, dtype=torch.float16).to("cuda").eval()
_siglip_proc = AutoProcessor.from_pretrained(SIGLIP_ID)


@torch.no_grad()
def embed_text(text):
    inp = _siglip_proc(text=[text], padding="max_length", max_length=64, truncation=True, return_tensors="pt").to("cuda")
    out = _siglip.get_text_features(**inp)
    out = out if torch.is_tensor(out) else out.pooler_output
    return torch.nn.functional.normalize(out.float(), dim=-1)[0].cpu().numpy()


B = QwenAgent(QWEN_B, QWEN_B_4BIT)
print("B:", B.name, f"| {torch.cuda.memory_allocated() / 1e9:.1f} GB allocated")

## 6. The pipeline with memory and stopping

`run_question(s, tau, variant)` runs one question. The variants are the full method and the ablations:

| variant | summary | close-ups | cluster rank | rounds |
|---|---|---|---|---|
| `full` | yes | yes | size × relevance | up to 3 |
| `size_rank` (B6) | yes | yes | size only (Flash-VStream style) | up to 3 |
| `summary_only` (B3) | yes | no | - | 1 |
| `closeups_only` (B4) | no | yes | size × relevance | 1 |

B5 (full method, round 1 only) needs no separate run: it is round 1 of `full`.

In [ ]:
VARIANTS = {
    "full":          dict(summary=True,  closeups=True,  rank="size_x_rel", rounds=MAX_ROUNDS),
    "size_rank":     dict(summary=True,  closeups=True,  rank="size",       rounds=MAX_ROUNDS),
    "summary_only":  dict(summary=True,  closeups=False, rank="size_x_rel", rounds=1),
    "closeups_only": dict(summary=False, closeups=True,  rank="size_x_rel", rounds=1),
}
_ROUTES = {}


def route_cached(s):
    path = RESULTS_DIR / f"router__{B.name}.json"
    cache = _ROUTES.setdefault("c", json.loads(path.read_text()) if path.exists() else {})
    if s["qid"] not in cache:
        cache[s["qid"]] = list(B.route(s["question"]))
        path.write_text(json.dumps(cache, indent=0))
    return cache[s["qid"]]


def parse_look_for(note):
    m = re.search(r"LOOK FOR:\s*(.+)", note or "", flags=re.I)
    return m.group(1).strip()[:200] if m else None


def run_question(s, tau, variant="full", force_all_rounds=False):
    v = VARIANTS[variant]
    B.usage = new_usage()
    t0 = time.time()
    task, route_conf = route_cached(s)
    route_usage = dict(B.usage)
    cl = clusters(s)
    summary = list(cl["reps"]) if v["summary"] else []
    n1 = CLOSEUPS_SHORT if s["span_s"] < SHORT_CLIP_S else CLOSEUPS_LONG
    closeups = pick_closeups(s, task, n1, rank=v["rank"]) if v["closeups"] else []
    memory, look_for, rounds, stop = None, None, [], None
    n_opt = len(s["options"])
    for r in range(v["rounds"]):
        parts = build_parts(s, summary, closeups, memory)
        b = B.choose(mc_system(s), parts + [{"type": "text", "text": mc_text(s)}], n_opt)
        stable = STABILITY_STOP and r > 0 and b["pred"] == rounds[-1]["pred"]
        rec = dict(round=r, summary=summary, closeups=list(closeups),
                   closeup_times=[s["frames"][i]["t"] for i in closeups],
                   closeup_clusters=[int(cl["label"][i]) for i in closeups],
                   pred=b["pred"], conf=round(b["conf"], 4), probs=b["probs"], letter_mass=b["letter_mass"],
                   stable=bool(stable), usage=dict(B.usage), payload_bytes=parts_bytes(parts),
                   distinct_frames=len(set(summary) | set(closeups)))
        rounds.append(rec)
        if stop is None and (b["conf"] >= tau or stable):
            stop = r
        more = v["closeups"] and len(closeups) < min(CLOSEUP_CAP, len(s["frames"]))
        if (stop is not None and not force_all_rounds) or r == v["rounds"] - 1 or not more:
            break
        # not confident: B writes its memory note, then new close-ups are chosen (steered by the LOOK FOR line)
        memory = B.generate(mc_system(s), parts + [{"type": "text", "text": mc_text(s)},
                                                    {"type": "text", "text": MEMORY_PROMPT}], max_new_tokens=220)
        look_for = parse_look_for(memory)
        new = pick_closeups(s, task, min(CLOSEUPS_PER_ROUND, CLOSEUP_CAP - len(closeups)), sent=closeups,
                            look_for_emb=embed_text(look_for) if look_for else None, rank=v["rank"])
        rec.update(memory=memory, look_for=look_for, new_closeups=new)
        closeups = sorted(set(closeups) | set(new))
    stop = len(rounds) - 1 if stop is None else stop
    fin = rounds[stop]
    return dict(qid=s["qid"], split=s["split"], dataset=s["dataset"], category=s["category"], task=s["task"],
                variant=variant, pred_task=task, route_conf=route_conf, router_correct=task == s["category"], tau=tau,
                span_s=round(s["span_s"], 2), n_frames_available=len(s["frames"]), K=len(cl["reps"]),
                answer=s["answer"], pred=fin["pred"], correct=fin["pred"] == s["answer"],
                round1_pred=rounds[0]["pred"], round1_correct=rounds[0]["pred"] == s["answer"],
                stop_round=stop, n_rounds_run=len(rounds), n_summary=len(summary), n_closeups=len(fin["closeups"]),
                distinct_frames=fin["distinct_frames"], usage=fin["usage"], route_usage=route_usage,
                payload_bytes=fin["payload_bytes"], rounds=rounds, seconds=round(time.time() - t0, 2), model=B.name)


def run_uniform(s, k, box):
    # baselines B1 / B2: B alone, k uniformly spaced frames at one resolution, one pass
    B.usage = new_usage()
    idx = uniform_idx(len(s["frames"]), k)
    parts = [{"type": "text", "text": f"The following {len(idx)} frames are sampled from the video in chronological order."}]
    for i in idx:
        parts += [{"type": "text", "text": f"Frame at {s['frames'][i]['t']:.1f}s:"},
                  {"type": "image", "path": s["frames"][i]["path"], "box": box}]
    b = B.choose(mc_system(s), parts + [{"type": "text", "text": mc_text(s)}], len(s["options"]))
    return dict(qid=s["qid"], split=s["split"], dataset=s["dataset"], category=s["category"], task=s["task"],
                n_frames=len(idx), frames=idx, answer=s["answer"], pred=b["pred"], conf=round(b["conf"], 4),
                correct=b["pred"] == s["answer"], usage=dict(B.usage), payload_bytes=parts_bytes(parts), model=B.name)

## 7. Runner (resumable JSONL)

In [ ]:
def load_jsonl(path):
    if not Path(path).exists():
        return pd.DataFrame()
    last = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.strip():
            r = json.loads(line)
            last[r["qid"]] = r
    return pd.DataFrame(list(last.values()))


def run_split(fn, samples, path, desc):
    prev = load_jsonl(path)
    done = set() if prev.empty else set(prev.qid[prev["error"].isna()])
    todo = [s for s in samples if s["qid"] not in done]
    print(f"{Path(path).name}: {len(done)} done, {len(todo)} to run")
    with open(path, "a", encoding="utf-8") as f:
        for s in tqdm(todo, desc=desc):
            try:
                rec = fn(s)
                rec["error"] = None
            except torch.cuda.OutOfMemoryError as e:
                torch.cuda.empty_cache()
                rec = dict(qid=s["qid"], dataset=s["dataset"], category=s["category"], error=f"CUDA OOM: {e}"[:400])
            except Exception as e:
                rec = dict(qid=s["qid"], dataset=s["dataset"], category=s["category"], error=repr(e)[:400])
            f.write(json.dumps(rec, ensure_ascii=False, default=float) + "\n")
            f.flush()
    return load_jsonl(path)


def path_for(name):
    return RESULTS_DIR / f"{B.name}__{name}.jsonl"

## 8. Calibrate τ (calibration split only)

The full method runs on the calibration questions with all rounds forced. Every τ on a grid is then replayed
offline, with the same rule as the live run: stop at the first round where confidence ≥ τ or (from round 2) the
answer is unchanged; otherwise stop at the last round. The cheapest τ within `TAU_TOLERANCE` of the best
accuracy is saved to `results/two_tier/tau__<model>.json`.

In [ ]:
calib = run_split(lambda s: run_question(s, tau=2.0, variant="full", force_all_rounds=True), CALIB,
                  path_for("calib__full-allrounds"), "calibration")


def replay(df, tau):
    rows = []
    for r in df[df.error.isna()].to_dict("records"):
        rr = r["rounds"]
        st = next((x["round"] for x in rr if x["conf"] >= tau or x["stable"]), len(rr) - 1)
        x = rr[st]
        rows.append(dict(correct=x["pred"] == r["answer"], rounds=st + 1, closeups=len(x["closeups"]),
                         tokens=x["usage"]["in_tokens"], images=x["usage"]["images"]))
    d = pd.DataFrame(rows)
    return dict(tau=tau, accuracy=100 * d.correct.mean(), rounds=d.rounds.mean(), closeups=d.closeups.mean(),
                tokens=d.tokens.mean(), images=d.images.mean())


grid = pd.DataFrame([replay(calib, t) for t in np.round(np.arange(0.30, 1.001, 0.025), 3)])
best = grid.accuracy.max()
TAU = float(grid[grid.accuracy >= best - 100 * TAU_TOLERANCE].sort_values("tokens").iloc[0].tau)
(RESULTS_DIR / f"tau__{B.name}.json").write_text(json.dumps(dict(tau=TAU, n_calib=int(calib.error.isna().sum()),
                                                                grid=grid.round(3).to_dict("records")), indent=1))
print(f"calibrated tau = {TAU} (best calibration accuracy {best:.1f}%)")
grid.round(2)

## 9. Full method on the test split

In [ ]:
tau_file = RESULTS_DIR / f"tau__{B.name}.json"
TAU = json.loads(tau_file.read_text())["tau"] if tau_file.exists() else TAU_DEFAULT
full = run_split(lambda s: run_question(s, tau=TAU, variant="full"), TEST, path_for(f"test__full__tau{TAU:g}"), "full method")
ok = full[full.error.isna()]
print(f"tau={TAU} | accuracy {100 * ok.correct.mean():.1f}% | round-1 accuracy {100 * ok.round1_correct.mean():.1f}% | "
      f"router {100 * ok.router_correct.mean():.1f}% | errors {full.error.notna().sum()}")

## 10. Baselines

| id | baseline |
|---|---|
| B1 | B alone, 32 uniform high-res frames (640x360), one pass |
| B2 | B alone, 64 uniform low-res frames (320x180), one pass |
| B3 | summary only (round 1, no close-ups) |
| B4 | close-ups only (round 1, no summary) |
| B5 | full method, round 1 only: taken from the `full` run, no extra compute |
| B6 | full method with clusters ranked by size only (Flash-VStream style), same τ |

In [ ]:
base = {
    "B1 uniform 32 hi-res": run_split(lambda s: run_uniform(s, 32, CLOSEUP_BOX), TEST, path_for("test__B1_uniform32_hires"), "B1"),
    "B2 uniform 64 lo-res": run_split(lambda s: run_uniform(s, 64, SUMMARY_BOX), TEST, path_for("test__B2_uniform64_lores"), "B2"),
    "B3 summary only": run_split(lambda s: run_question(s, TAU, "summary_only"), TEST, path_for("test__B3_summary_only"), "B3"),
    "B4 close-ups only": run_split(lambda s: run_question(s, TAU, "closeups_only"), TEST, path_for("test__B4_closeups_only"), "B4"),
    "B6 size-ranked": run_split(lambda s: run_question(s, TAU, "size_rank"), TEST, path_for(f"test__B6_size_rank__tau{TAU:g}"), "B6"),
}

## 11. Results

In [ ]:
def summarize_run(name, df, correct_col="correct"):
    d = df[df.error.isna()]
    tok = d.usage.map(lambda u: u["in_tokens"])
    img = d.usage.map(lambda u: u["images"])
    frames = d["distinct_frames"] if "distinct_frames" in d else d["n_frames"]
    return dict(method=name, n=len(d), accuracy=100 * d[correct_col].mean(), frames_seen=frames.mean(),
                image_inputs=img.mean(), input_tokens=tok.mean(), KB_sent=d.payload_bytes.mean() / 1024)


rows = [summarize_run("PROPOSED (full, up to 3 rounds)", ok)]
b5 = ok.copy()
b5["distinct_frames"] = b5.rounds.map(lambda r: r[0]["distinct_frames"])
b5["usage"] = b5.rounds.map(lambda r: r[0]["usage"])
b5["payload_bytes"] = b5.rounds.map(lambda r: r[0]["payload_bytes"])
for name, df in base.items():
    rows.append(summarize_run(name, df))
rows.append(summarize_run("B5 full, round 1 only", b5, "round1_correct"))
for label, fname in [("nb04 proposed (A notes -> B)", None), ("nb04 B + uniform (matched)", "__baseline-B-uniform")]:
    hits = sorted(NB04_DIR.glob(f"*__test__tau*{fname or ''}.jsonl"))
    hits = [h for h in hits if (fname is not None) == ("baseline" in h.name) and ("adaptive" not in h.name)]
    if hits:
        d = load_jsonl(hits[0])
        d = d[d.error.isna()]
        rows.append(dict(method=label, n=len(d), accuracy=100 * d.correct.mean()))
summary_table = pd.DataFrame(rows).set_index("method")
print(f"Test split, Qwen3-VL-4B, tau = {TAU}")
summary_table.round(1)

In [ ]:
# Accuracy per dataset / category for every method
cols = {"PROPOSED": ok.set_index("qid").correct, "B5 round-1": ok.set_index("qid").round1_correct}
for name, df in base.items():
    cols[name.split(" ")[0]] = df[df.error.isna()].set_index("qid").correct
acc = pd.DataFrame(cols).astype(float)
meta = ok.set_index("qid")[["dataset", "category"]]
acc = meta.join(acc)
per_cat = 100 * acc.groupby(["dataset", "category"]).mean(numeric_only=True)
per_cat.insert(0, "n", acc.groupby(["dataset", "category"]).size())
per_ds = 100 * acc.groupby("dataset").mean(numeric_only=True)
display(per_cat.round(1))
display(per_ds.round(1))

In [ ]:
from math import comb


def mcnemar(x, y):
    # exact two-sided McNemar test on paired correct/incorrect outcomes
    x, y = np.asarray(x, bool), np.asarray(y, bool)
    b, c = int((x & ~y).sum()), int((~x & y).sum())
    n = b + c
    p = min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n) if n else 1.0
    return b, c, p


print("PROPOSED vs each baseline (paired): wins / losses / p-value")
for name in acc.columns[3:]:
    m = acc[["PROPOSED", name]].dropna()
    print(f"  vs {name:22s} {mcnemar(m.PROPOSED, m[name])}")

print("\nStopping behaviour (full method)")
display(ok.groupby("stop_round").agg(n=("qid", "size"), accuracy=("correct", "mean"),
                                     closeups=("n_closeups", "mean"), seconds=("seconds", "mean")).round(3))
fixed = sum((not r[0]["pred"] == a) and (x == a) for r, a, x in zip(ok.rounds, ok.answer, ok.pred))
broken = sum((r[0]["pred"] == a) and (x != a) for r, a, x in zip(ok.rounds, ok.answer, ok.pred))
print(f"extra rounds fixed {fixed} answers and broke {broken}")
print(f"router accuracy {100 * ok.router_correct.mean():.1f}%")

In [ ]:
# One question end to end: task, clusters, close-ups per round, memory notes
r = ok.sample(1).iloc[0].to_dict()
s = BY_QID[r["qid"]]
print(f"[{r['dataset']}/{r['category']} -> {r['pred_task']}] {s['question']}")
for i, o in enumerate(s["options"]):
    print(f"   ({LETTERS[i]}) {o}")
print(f"{r['n_frames_available']} frames, K={r['K']} clusters | answer {r['answer']} | predicted {r['pred']} "
      f"(stop round {r['stop_round']})")
for x in r["rounds"]:
    print(f"  round {x['round'] + 1}: summary {len(x['summary'])} + close-ups {len(x['closeups'])} "
          f"{[round(t) for t in x['closeup_times']]} -> {x['pred']} ({x['conf']:.2f}){' [stable]' if x['stable'] else ''}")
    if x.get("memory"):
        print("     memory:", x["memory"][:400].replace("\n", " | "))

## 12. Embedding visualization

Each frame's SigLIP2 embedding is projected to 2D.
- **Left:** all frames plus the **summary** frames (one per cluster).
- **Right:** all frames plus the **close-ups**, coloured by the round that sent them.
- `COLOR_BY_TIME = True` shades the background dots from early (light) to late (dark).
- `METHOD = "tsne"` needs scikit-learn (`pip install scikit-learn`); otherwise PCA is used.

In [ ]:
def project_2d(x, method="pca"):
    if method == "tsne":
        try:
            from sklearn.manifold import TSNE
            return TSNE(n_components=2, init="pca", perplexity=min(30, max(5, len(x) // 4)), random_state=0).fit_transform(x)
        except ImportError:
            print("scikit-learn not installed -> PCA")
    xc = x - x.mean(0)
    _, _, vt = np.linalg.svd(xc, full_matrices=False)
    return xc @ vt[:2].T


def plot_embedding(qid=None, results=None, method="pca", color_by_time=True, thumbs=True):
    results = ok if results is None else results
    r = results[results.qid == qid].iloc[0].to_dict() if qid else results.sample(1).iloc[0].to_dict()
    s = BY_QID[r["qid"]]
    xy = project_2d(s["feat"]["emb"], method)
    t = s["feat"]["t"]
    fin = r["rounds"][r["stop_round"]]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    bg = dict(c=t, cmap="Blues", vmin=t.min() - 0.3 * (t.max() - t.min() + 1e-6)) if color_by_time else dict(c="#5b7bd5")
    for ax in axes:
        ax.scatter(xy[:, 0], xy[:, 1], s=14, alpha=0.65, label="frame features", **bg)
        ax.set_xlabel("Principal Component 1" if method == "pca" else "t-SNE 1")
        ax.set_ylabel("Principal Component 2" if method == "pca" else "t-SNE 2")
    summ = fin["summary"]
    if summ:
        axes[0].scatter(xy[summ, 0], xy[summ, 1], s=70, c="red", edgecolors="k", linewidths=0.5, label="summary frames (1 per cluster)")
    axes[0].set_title(f"Summary memory: {len(summ)} of {len(t)} frames")
    styles = [("red", "o"), ("orange", "s"), ("green", "^")]
    seen = []
    for x in r["rounds"][: r["stop_round"] + 1]:
        new = [i for i in x["closeups"] if i not in seen]
        c, m = styles[x["round"] % 3]
        if new:
            axes[1].scatter(xy[new, 0], xy[new, 1], s=80, c=c, marker=m, edgecolors="k", linewidths=0.5,
                            label=f"close-ups, round {x['round'] + 1} ({len(new)})")
        seen += new
    axes[1].set_title(f"Close-up memory: {len(seen)} high-res frames")
    for ax in axes:
        ax.legend(loc="best", fontsize=8)
    ok_txt = "correct" if r["correct"] else f"wrong (answer {r['answer']})"
    fig.suptitle(f"{r['qid']} [{r['category']} -> {r['pred_task']}] K={r['K']} | predicted {r['pred']}, {ok_txt}, "
                 f"stopped after round {r['stop_round'] + 1}\n{' '.join(s['question'].split())[:140]}", fontsize=10)
    plt.tight_layout()
    plt.show()
    if thumbs:
        for title, idx, box in [("summary", summ, SUMMARY_BOX), ("close-ups", sorted(seen), CLOSEUP_BOX)]:
            idx = idx[:: max(1, len(idx) // 16)][:16]
            if not idx:
                continue
            f2, ax2 = plt.subplots(1, len(idx), figsize=(1.6 * len(idx), 1.4))
            for a, i in zip(np.atleast_1d(ax2), idx):
                a.imshow(load_frame(s["frames"][i]["path"], box))
                a.set_title(f"{s['frames'][i]['t']:.0f}s", fontsize=7)
                a.axis("off")
            f2.suptitle(title, fontsize=9, x=0.02, ha="left")
            plt.show()
    return r["qid"]


METHOD, COLOR_BY_TIME = "pca", True
plot_embedding(method=METHOD, color_by_time=COLOR_BY_TIME)

In [ ]:
# Pick a specific question, e.g. a long Video-MME video that needed several rounds
multi = ok[(ok.dataset == "videomme") & (ok.stop_round > 0)]
if len(multi):
    plot_embedding(multi.sample(1).qid.iloc[0], method=METHOD, color_by_time=COLOR_BY_TIME)

---
# V2 - Round 1 only, larger summary, uniform frames for counting

Everything above stays as it is. This section tests the three changes the V1 results pointed to, **without**
touching the V1 configuration or results:

| Change | Why (from the V1 run) |
|---|---|
| **One round only** | 93% of questions already stopped after round 1, and extra rounds fixed 3 answers but broke 5 |
| **Bigger summary** (`K_SCALE` 8 → 12, `K_MAX` 48 → 56) | The wins came from whole-video coverage (Video-MME temporal +14, spatial +11), and low-res frames cost ~66 tokens vs ~220 |
| **Uniform close-ups for counting** | Clustering and gap filling both lost to uniform spacing on counting (MVBench 51 vs 63, Video-MME 40 vs 49) |

**Runs in this section** (each is a single pass per question, so they are cheap):

| Run | Summary K | Counting close-ups | Purpose |
|---|---|---|---|
| `v2_full` | large | uniform | both changes together |
| `v2_bigK` | large | gap filling (V1) | effect of the bigger summary alone |
| `v2_count_uniform` | V1 size | uniform | effect of the counting change alone |
| `B7 uniform mixed` | - | - | same token budget as `v2_full`, but **uniform** low-res + high-res frames and **no clustering** |

`B7` is the important control: it answers whether the clustering matters, or whether the gain is just "low-res
coverage plus a few high-res frames".

In [ ]:
# ===== V2 configuration =============================================================================
V2_K_SCALE, V2_K_MAX = 12.0, 56            # V1: 8.0, 48  ->  5 min: 49 clusters (was 40), 15 min: 56 (was 48)
V2_CLOSEUPS_SHORT, V2_CLOSEUPS_LONG = 8, 16
V2_RUNS = {
    "v2_full":          dict(k_scale=V2_K_SCALE, k_max=V2_K_MAX, counting_uniform=True),
    "v2_bigK":          dict(k_scale=V2_K_SCALE, k_max=V2_K_MAX, counting_uniform=False),
    "v2_count_uniform": dict(k_scale=K_SCALE,    k_max=K_MAX,    counting_uniform=True),
}

In [ ]:
# ===== V2: reload the model if the "free the GPU" cell was already run =============================
if "B" not in globals():
    B = QwenAgent(QWEN_B, QWEN_B_4BIT)
    print("reloaded", B.name)
else:
    print("using the model already loaded:", B.name)

In [ ]:
# ===== V2 clustering and close-up selection (V1 functions are left untouched) =======================
def clusters_k(s, k_scale, k_max):
    k = round(k_scale * math.log2(1 + s["span_s"] / 10.0))
    k = int(max(1, min(np.clip(k, K_MIN, k_max), len(s["frames"]) // 2)))
    key = f"_cl{k}"
    if key not in s:
        sig = signals(s)
        seed = int(hashlib.md5(s["frame_dir"].encode()).hexdigest()[:8], 16) ^ SEED
        lab, cent = kmeans(sig["emb"], k, seed)
        members, reps = [], []
        for j in range(k):
            m = np.flatnonzero(lab == j)
            if not len(m):
                continue
            near = m[np.argsort(-(sig["emb"][m] @ cent[j]))[:3]]
            reps.append(int(near[np.argmax(sig["q"][near])]))
            members.append(m)
        order = np.argsort([s["frames"][r]["t"] for r in reps])
        members, reps = [members[i] for i in order], [reps[i] for i in order]
        label = np.full(sig["n"], -1)
        for j, m in enumerate(members):
            label[m] = j
        s[key] = dict(members=members, reps=reps, label=label, size=np.array([len(m) for m in members]),
                      seconds=np.array([len(m) / s["fps"] for m in members]))
    return s[key]


def pick_closeups_v2(s, task, n, cl, counting_uniform):
    sig = signals(s)
    if task == "counting" and counting_uniform:
        return uniform_idx(sig["n"], n)                       # uniform beat both other rules on counting in V1
    rel = sig["rel"]
    avail = np.ones(sig["n"], dtype=bool)
    soft = avail.copy()
    soft[cl["reps"]] = False
    if soft.sum() >= n:
        avail = soft
    n = int(min(n, avail.sum()))
    picked = []
    if task in TIME_TASKS:                                     # temporal (and counting when counting_uniform=False)
        w = (0.5, 0.3, 0.2) if task == "counting" else (0.0, 0.6, 0.4)
        score = w[0] * sig["motion"] + w[1] * rel + w[2] * sig["q"]
        for _ in range(n):
            bounds = [-1] + sorted(picked) + [sig["n"]]
            for a, b in sorted(zip(bounds[:-1], bounds[1:]), key=lambda ab: -(ab[1] - ab[0])):
                inside = [i for i in range(a + 1, b) if avail[i] and i not in picked]
                if not inside:
                    continue
                lo, hi = a + (b - a) / 4, b - (b - a) / 4
                picked.append(max([i for i in inside if lo <= i <= hi] or inside, key=lambda i: score[i]))
                break
    else:
        size = cl["size"] / cl["size"].max()
        cscore = size * np.array([rel[m].max() for m in cl["members"]])
        count = np.zeros(len(cl["members"]), dtype=int)
        fscore = rel + 0.3 * sig["q"]
        while len(picked) < n:
            for j in sorted(range(len(cl["members"])), key=lambda j: (count[j], -cscore[j])):
                cand = [i for i in cl["members"][j] if avail[i] and i not in picked]
                if cand:
                    picked.append(int(max(cand, key=lambda i: fscore[i])))
                    count[j] += 1
                    break
            else:
                break
    return sorted(int(i) for i in picked)


def build_parts_k(s, cl, summary, closeups, scene_labels=True):
    parts = []
    if summary:
        rep_sec = {r: cl["seconds"][j] for j, r in enumerate(cl["reps"])} if scene_labels else {}
        parts.append({"type": "text", "text": f"Overview of the whole video: {len(summary)} low-resolution frames, "
                                              + ("one per distinct scene, " if scene_labels else "sampled uniformly, ")
                                              + "in chronological order."})
        for i in summary:
            tag = (f"Scene at {s['frames'][i]['t']:.1f}s (visible ~{rep_sec.get(i, 0):.0f}s in total):"
                   if scene_labels else f"Frame at {s['frames'][i]['t']:.1f}s:")
            parts += [{"type": "text", "text": tag},
                      {"type": "image", "path": s["frames"][i]["path"], "box": SUMMARY_BOX}]
    if closeups:
        head = "Close-up frames at higher resolution" if summary else "Frames sampled from the video"
        parts.append({"type": "text", "text": f"{head}, in chronological order:"})
        for i in closeups:
            parts += [{"type": "text", "text": f"Frame at {s['frames'][i]['t']:.1f}s:"},
                      {"type": "image", "path": s["frames"][i]["path"], "box": CLOSEUP_BOX}]
    return parts


def _record(s, task, route_conf, summary, closeups, b, parts, t0, run, K):
    rec = dict(round=0, summary=list(summary), closeups=list(closeups),
               closeup_times=[s["frames"][i]["t"] for i in closeups], closeup_clusters=[],
               pred=b["pred"], conf=round(b["conf"], 4), probs=b["probs"], letter_mass=b["letter_mass"],
               stable=False, usage=dict(B.usage), payload_bytes=parts_bytes(parts),
               distinct_frames=len(set(summary) | set(closeups)))
    return dict(qid=s["qid"], split=s["split"], dataset=s["dataset"], category=s["category"], task=s["task"],
                variant=run, pred_task=task, route_conf=route_conf, router_correct=task == s["category"], tau=None,
                span_s=round(s["span_s"], 2), n_frames_available=len(s["frames"]), K=K, answer=s["answer"],
                pred=b["pred"], correct=b["pred"] == s["answer"], round1_pred=b["pred"],
                round1_correct=b["pred"] == s["answer"], stop_round=0, n_rounds_run=1, n_summary=len(summary),
                n_closeups=len(closeups), distinct_frames=rec["distinct_frames"], usage=dict(B.usage),
                payload_bytes=rec["payload_bytes"], rounds=[rec], seconds=round(time.time() - t0, 2), model=B.name)


def run_v2(s, cfg, run="v2"):
    B.usage = new_usage()
    t0 = time.time()
    task, route_conf = route_cached(s)
    cl = clusters_k(s, cfg["k_scale"], cfg["k_max"])
    n1 = V2_CLOSEUPS_SHORT if s["span_s"] < SHORT_CLIP_S else V2_CLOSEUPS_LONG
    summary, closeups = list(cl["reps"]), pick_closeups_v2(s, task, n1, cl, cfg["counting_uniform"])
    parts = build_parts_k(s, cl, summary, closeups)
    b = B.choose(mc_system(s), parts + [{"type": "text", "text": mc_text(s)}], len(s["options"]))
    return _record(s, task, route_conf, summary, closeups, b, parts, t0, run, len(cl["reps"]))


def run_uniform_mixed(s, cfg, run="B7"):
    # same token budget as v2_full, but uniform low-res + high-res frames and no clustering
    B.usage = new_usage()
    t0 = time.time()
    task, route_conf = route_cached(s)
    n_lo = len(clusters_k(s, cfg["k_scale"], cfg["k_max"])["reps"])
    n_hi = V2_CLOSEUPS_SHORT if s["span_s"] < SHORT_CLIP_S else V2_CLOSEUPS_LONG
    summary, closeups = uniform_idx(len(s["frames"]), n_lo), uniform_idx(len(s["frames"]), n_hi)
    parts = build_parts_k(s, dict(reps=summary), summary, closeups, scene_labels=False)
    b = B.choose(mc_system(s), parts + [{"type": "text", "text": mc_text(s)}], len(s["options"]))
    return _record(s, task, route_conf, summary, closeups, b, parts, t0, run, n_lo)

In [ ]:
# ===== V2 runs ======================================================================================
v2 = {name: run_split(lambda s, c=cfg, n=name: run_v2(s, c, n), TEST, path_for(f"test__{name}"), name)
      for name, cfg in V2_RUNS.items()}
v2["B7_uniform_mixed"] = run_split(lambda s: run_uniform_mixed(s, V2_RUNS["v2_full"]), TEST,
                                   path_for("test__B7_uniform_mixed"), "B7")
for name, df in v2.items():
    d = df[df.error.isna()]
    print(f"{name:18s} accuracy {100 * d.correct.mean():5.1f}%  | summary {d.n_summary.mean():4.1f} + close-ups "
          f"{d.n_closeups.mean():4.1f} | tokens {d.usage.map(lambda u: u['in_tokens']).mean():6.0f} | errors {df.error.notna().sum()}")

## V2 - Results: everything side by side

In [ ]:
# ===== V2 comparison ================================================================================
def run_stats(name, df, correct_col="correct"):
    d = df[df.error.isna()]
    frames = d["distinct_frames"] if "distinct_frames" in d else d["n_frames"]
    return dict(method=name, n=len(d), accuracy=100 * d[correct_col].mean(),
                mvbench=100 * d[d.dataset == "mvbench"][correct_col].mean(),
                videomme=100 * d[d.dataset == "videomme"][correct_col].mean(),
                frames=frames.mean(), tokens=d.usage.map(lambda u: u["in_tokens"]).mean(),
                images=d.usage.map(lambda u: u["images"]).mean(), KB=d.payload_bytes.mean() / 1024,
                seconds=d.seconds.mean() if "seconds" in d else np.nan)


table = [run_stats("V2 full (1 round, big K, uniform counting)", v2["v2_full"]),
         run_stats("V2 big K only", v2["v2_bigK"]),
         run_stats("V2 uniform counting only", v2["v2_count_uniform"]),
         run_stats("B7 uniform mixed (no clustering)", v2["B7_uniform_mixed"]),
         run_stats("V1 full (up to 3 rounds)", ok),
         run_stats("V1 round 1 only", ok, "round1_correct")]
for name, df in base.items():
    table.append(run_stats(name, df))
v2_table = pd.DataFrame(table).set_index("method").round(1)
print("Test split, Qwen3-VL-4B")
v2_table

In [ ]:
# ===== V2 per-category and significance =============================================================
cols = {"V2_full": v2["v2_full"], "V2_bigK": v2["v2_bigK"], "V2_count_uni": v2["v2_count_uniform"],
        "B7_uniform_mixed": v2["B7_uniform_mixed"], "B1_uniform32": base["B1 uniform 32 hi-res"]}
acc2 = pd.DataFrame({k: d[d.error.isna()].set_index("qid").correct.astype(float) for k, d in cols.items()})
acc2["V1_round1"] = ok.set_index("qid").round1_correct.astype(float)
acc2 = ok.set_index("qid")[["dataset", "category"]].join(acc2)
per_cat2 = 100 * acc2.groupby(["dataset", "category"]).mean(numeric_only=True)
per_cat2.insert(0, "n", acc2.groupby(["dataset", "category"]).size())
display(per_cat2.round(1))
display((100 * acc2.groupby("dataset").mean(numeric_only=True)).round(1))

print("V2 full vs each (wins, losses, p):")
for name in [c for c in acc2.columns if c not in ("dataset", "category", "V2_full")]:
    m = acc2[["V2_full", name]].dropna()
    print(f"  vs {name:18s} {mcnemar(m.V2_full, m[name])}")

In [ ]:
# ===== V2 embedding plot ============================================================================
plot_embedding(results=v2["v2_full"], method=METHOD, color_by_time=COLOR_BY_TIME)

In [ ]:
# Free the GPU when done
B.close()
del B, _siglip
gc.collect(); torch.cuda.empty_cache()